## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
 import sys

def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    it = iter(data)
    n = next(it)
    A = next(it)
    B = next(it)
    p = [next(it) for _ in range(n)]
    pos = [0] * n
    for i, v in enumerate(p):
        pos[v] = i

    answer = []


    def use_swap_magic():
        answer.append(0)
        i, j = pos[A], pos[B]
        p[i], p[j] = p[j], p[i]
        pos[A], pos[B] = j, i

    def use_add_magic(v):
        v %= n
        if v < 0:
            v += n
        if v == 0:
            return
        answer.append(v)
        new_p = [(x + v) % n for x in p]
        for i, val in enumerate(new_p):
            pos[val] = i
        p[:] = new_p

    def use_xor_magic(v):
        if v == 0:
            return
        answer.append(-v)
        new_p = [x ^ v for x in p]
        for i, val in enumerate(new_p):
            pos[val] = i
        p[:] = new_p


    lowbitLen = (A - B + n) % n
    lowbitLen &= -lowbitLen
    if lowbitLen == 0:
        lowbitLen = n


    def get_pair_pos(x, y):
        delta = (y - x + n - lowbitLen + n) % n
        px = py = 0
        step = n // 2
        while step >= 2 * lowbitLen:
            if delta >= step:
                delta -= step
                py += step // 2
            else:
                px += step // 2
            step >>= 1
        px += n // 2
        px += (x & (lowbitLen - 1))
        py += (x & (lowbitLen - 1))
        return px, py

    def apply_swap(x, y):
        if (x // lowbitLen) % 2 == (y // lowbitLen) % 2:
            if (x // lowbitLen) % 2 == 0:
                mid = (x & (lowbitLen - 1)) + lowbitLen
            else:
                mid = (x & (lowbitLen - 1))
            apply_swap(x, mid)
            apply_swap(y, mid)
            apply_swap(x, mid)
            return

        pA, pB = get_pair_pos(A, B)
        pX, pY = get_pair_pos(x, y)

        use_add_magic((pX - x + n) % n)
        use_xor_magic(pX ^ pA)
        use_add_magic((A - pA + n) % n)
        use_swap_magic()
        use_add_magic((pA - A + n) % n)
        use_xor_magic(pX ^ pA)
        use_add_magic((x - pX + n) % n)


    def build_permutation(val, length):
        if length == 1:
            return []
        half = length // 2
        left_val = [0] * half
        right_val = [0] * half
        for i in range(half):
            left_val[i] = val[i * 2] // 2
            right_val[i] = val[i * 2 + 1] // 2

        left_ops = build_permutation(left_val, half)
        if left_ops is None:
            return None
        right_ops = build_permutation(right_val, half)
        if right_ops is None:
            return None

        ops = []
        if val[0] & 1:
            ops.append(1 if length == 2 else -1)

        cur_xor_l = 0
        for x in left_ops:
            if x > 0:
                ops.append(-1)
                ops.append(1)
            else:
                ops.append(x * 2)
                cur_xor_l ^= (-x) * 2
        if cur_xor_l:
            ops.append(-cur_xor_l)

        cur_xor_r = 0
        for x in right_ops:
            if x > 0:
                ops.append(1)
                ops.append(-1)
            else:
                ops.append(x * 2)
                cur_xor_r ^= (-x) * 2

        if (cur_xor_r & half) != (cur_xor_l & half):
            return None
        if cur_xor_l >= half:
            cur_xor_l -= half
        if cur_xor_r >= half:
            cur_xor_r -= half
        if cur_xor_l != cur_xor_r:
            return None

        # 合并相邻的负数操作
        merged = []
        for x in ops:
            if not merged:
                merged.append(x)
            else:
                if x < 0 and merged[-1] < 0:
                    merged[-1] = -(( -merged[-1] ) ^ ( -x ))
                    if merged[-1] == 0:
                        merged.pop()
                else:
                    merged.append(x)
        return merged

    if lowbitLen > 1:
        base_val = [0] * lowbitLen
        for i in range(n):
            base_val[i % lowbitLen] = p[i] & (lowbitLen - 1)
        ops = build_permutation(base_val, lowbitLen)
        if ops is None:
            print(-1)
            return
        for x in ops:
            if x > 0:
                use_add_magic(x)
            else:
                use_xor_magic(-x)

    for rem in range(lowbitLen):
        vec = []
        for j in range(rem, n, lowbitLen):
            vec.append(p[j])
        vec.sort()
        ok = True
        ptr = 0
        for j in range(rem, n, lowbitLen):
            if vec[ptr] != j:
                ok = False
                break
            ptr += 1
        if not ok:
            print(-1)
            return
        for j in range(rem, n, lowbitLen):
            if p[j] != j:
                apply_swap(j, p[j])

    # 输出答案
    out = [str(len(answer))]
    for x in answer:
        if x == 0:
            out.append("0")
        elif x < 0:
            out.append(f"1 {-x}")
        else:
            out.append(f"2 {x}")
    sys.stdout.write("\n".join(out))

if __name__ == "__main__":
    main()

## B 长跑

In [ ]:
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define MAXN 2005          // 最大商店数（含起点、终点）
#define MAX_QUEUE 4000000  // 队列容量足够大

typedef struct {
    int p;      // 位置
    int c;      // 花费
} Shop;

typedef struct {
    int now;    // 当前位置（距离起点）
    int money;  // 剩余硬币
    int id;     // 当前商店在数组中的下标
} Node;

// 比较函数
int cmp(const void *a, const void *b) {
    Shop *sa = (Shop*)a;
    Shop *sb = (Shop*)b;
    if (sa->p != sb->p) return sa->p - sb->p;
    return sa->c - sb->c;
}

int main() {
    int N, L, Maxn, S;
    while (scanf("%d%d%d%d", &N, &L, &Maxn, &S) != EOF) {
        if (Maxn >= L) {
            printf("Yes\n");
            for (int i = 0; i < N; i++) {
                int p, c;
                scanf("%d%d", &p, &c);
            }
            continue;
        }

        Shop shops[MAXN];
        shops[0].p = 0;
        shops[0].c = 0;
        for (int i = 1; i <= N; i++) {
            scanf("%d%d", &shops[i].p, &shops[i].c);
        }
        shops[N + 1].p = L;
        shops[N + 1].c = 0;
        int total = N + 2;   // 起点 + N个商店 + 终点
        qsort(shops, total, sizeof(Shop), cmp);

        // BFS队列
        Node queue[MAX_QUEUE];
        int front = 0, rear = 0;
        Node start = {0, S, 0};   // 起点位置0，剩余S硬币，对应shops[0]
        queue[rear++] = start;
        int ok = 0;

        while (front < rear) {
            Node cur = queue[front++];
            if (cur.now == L) {
                ok = 1;
                break;
            }

            for (int i = cur.id + 1; i < total; i++) {
                int dist = shops[i].p - cur.now;
                if (dist > Maxn) break;   // 距离超过体力上限，后面的更远，直接退出
                if (cur.money >= shops[i].c) {
                    Node nxt;
                    nxt.now = shops[i].p;
                    nxt.money = cur.money - shops[i].c;
                    nxt.id = i;
                    queue[rear++] = nxt;
                    // 防止队列溢出（题目数据不会达到此上限）
                    if (rear >= MAX_QUEUE) rear = MAX_QUEUE - 1;
                }
            }
        }

        printf("%s\n", ok ? "Yes" : "No");
    }
    return 0;
}

## C 最长回文

In [ ]:
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

int main() {
    int n;
    char A[1000005], B[1000005];
    scanf("%d", &n);
    scanf("%s", A);
    scanf("%s", B);

    int lenA = strlen(A);
    int lenB = strlen(B);

    // 构建扩展字符串 t1 和半径数组 p1
    int len_t1 = 2 * lenA + 2;
    char *t1 = (char*)malloc((len_t1 + 1) * sizeof(char)); // +1 用于结尾 '\0'
    int *p1 = (int*)malloc(len_t1 * sizeof(int));

    t1[0] = '$';
    t1[1] = '#';
    int idx = 2;
    for (int i = 0; i < lenA; ++i) {
        t1[idx++] = A[i];
        t1[idx++] = '#';
    }
    t1[idx] = '\0';  // idx == len_t1

    int mx = 0, id = 0;
    for (int i = 1; i < len_t1; ++i) {
        if (i < mx)
            p1[i] = (p1[2 * id - i] < mx - i) ? p1[2 * id - i] : mx - i;
        else
            p1[i] = 1;
        while (i - p1[i] >= 0 && i + p1[i] < len_t1 && t1[i - p1[i]] == t1[i + p1[i]])
            p1[i]++;
        if (i + p1[i] > mx) {
            mx = i + p1[i];
            id = i;
        }
    }

    // 构建扩展字符串 t2 和半径数组 p2
    int len_t2 = 2 * lenB + 2;
    char *t2 = (char*)malloc((len_t2 + 1) * sizeof(char));
    int *p2 = (int*)malloc(len_t2 * sizeof(int));

    t2[0] = '$';
    t2[1] = '#';
    idx = 2;
    for (int i = 0; i < lenB; ++i) {
        t2[idx++] = B[i];
        t2[idx++] = '#';
    }
    t2[idx] = '\0';

    mx = 0, id = 0;
    for (int i = 1; i < len_t2; ++i) {
        if (i < mx)
            p2[i] = (p2[2 * id - i] < mx - i) ? p2[2 * id - i] : mx - i;
        else
            p2[i] = 1;
        while (i - p2[i] >= 0 && i + p2[i] < len_t2 && t2[i - p2[i]] == t2[i + p2[i]])
            p2[i]++;
        if (i + p2[i] > mx) {
            mx = i + p2[i];
            id = i;
        }
    }

    // 计算最长公共回文子串长度
    int L = len_t1;          // t1 的有效长度
    int ans = 1;
    for (int i = 2; i < L; ++i) {
        int tmp = (p1[i] > p2[i - 2]) ? p1[i] : p2[i - 2];
        // 扩展比较，同时确保 t2 索引不越界
        while (i - tmp >= 0 && i + tmp - 2 < L && i + tmp - 2 < len_t2 &&
                t1[i - tmp] == t2[i + tmp - 2]) {
            tmp++;
        }
        if (tmp > ans) ans = tmp;
    }

    printf("%d\n", ans - 1);

    free(t1);
    free(p1);
    free(t2);
    free(p2);
    return 0;
}

## D 优惠券

In [ ]:
#include <stdio.h>
#include <string.h>
#include <stdbool.h>

#define MAX_X 1000005
#define MAX_M 500005

int state[MAX_X];
int last_event[MAX_X];
bool visited[MAX_X];
int modified_coupons[MAX_X];
int mod_cnt;

int bit[MAX_M + 5];
int m;

void add(int idx, int delta) {
    while (idx <= m) {
        bit[idx] += delta;
        idx += idx & -idx;
    }
}

int sum(int idx) {
    int s = 0;
    while (idx > 0) {
        s += bit[idx];
        idx -= idx & -idx;
    }
    return s;
}

int find_kth(int k) {
    int idx = 0;
    int bit_mask = 1;
    while (bit_mask <= m) bit_mask <<= 1;
    bit_mask >>= 1;
    while (bit_mask) {
        int nxt = idx + bit_mask;
        if (nxt <= m && bit[nxt] < k) {
            k -= bit[nxt];
            idx = nxt;
        }
        bit_mask >>= 1;
    }
    return idx + 1;
}

int main() {
    while (scanf("%d", &m) == 1) {
        mod_cnt = 0;
        int error_line = -1;
        memset(bit, 0, sizeof(int) * (m + 2));

        for (int i = 1; i <= m; ++i) {
            char op[10];
            scanf("%s", op);
            int x = 0;
            if (op[0] == '0') op[0] = 'O';   // 处理输入中 '0' 代表 'O' 的情况
            if (op[0] == 'I' || op[0] == 'O') {
                scanf("%d", &x);
            }

            if (error_line != -1) continue;

            if (op[0] == '?') {
                add(i, 1);
            } else if (op[0] == 'I') {
                if (!visited[x]) {
                    visited[x] = true;
                    modified_coupons[mod_cnt++] = x;
                }
                if (state[x] == 1) {
                    int cnt = (last_event[x] > 0) ? sum(last_event[x] - 1) : 0;
                    int pos = find_kth(cnt + 1);
                    if (pos >= 1 && pos <= m && pos < i) {
                        add(pos, -1);
                    } else {
                        error_line = i;
                    }
                }
                if (error_line == -1) {
                    state[x] = 1;
                    last_event[x] = i;
                }
            } else if (op[0] == 'O') {
                if (!visited[x]) {
                    visited[x] = true;
                    modified_coupons[mod_cnt++] = x;
                }
                if (state[x] == 0) {
                    int cnt = (last_event[x] > 0) ? sum(last_event[x] - 1) : 0;
                    int pos = find_kth(cnt + 1);
                    if (pos >= 1 && pos <= m && pos < i) {
                        add(pos, -1);
                    } else {
                        error_line = i;
                    }
                }
                if (error_line == -1) {
                    state[x] = 0;
                    last_event[x] = i;
                }
            } else {
                // 防御：未知字符当作 '?'
                add(i, 1);
            }
        }

        printf("%d\n", error_line);

        for (int i = 0; i < mod_cnt; ++i) {
            int x = modified_coupons[i];
            state[x] = 0;
            last_event[x] = 0;
            visited[x] = false;
        }
    }
    return 0;
}

## E 任意点

In [ ]:
#include <stdio.h>

#define MAXN 105

int father[MAXN];
int x[MAXN], y[MAXN];

int find(int x) {
    if (father[x] == x) return x;
    return father[x] = find(father[x]);
}

void merge(int a, int b) {
    int fa = find(a);
    int fb = find(b);
    if (fa != fb) father[fb] = fa;
}

int main() {
    int n;
    while (scanf("%d", &n) == 1 && n != 0) {
        for (int i = 1; i <= n; i++) {
            father[i] = i;
            scanf("%d%d", &x[i], &y[i]);
        }

        for (int i = 1; i <= n; i++) {
            for (int j = i + 1; j <= n; j++) {
                if (x[i] == x[j] || y[i] == y[j]) {
                    merge(i, j);
                }
            }
        }

        int cnt = 0;
        for (int i = 1; i <= n; i++) {
            if (father[i] == i) cnt++;
        }

        printf("%d\n", cnt - 1);
    }
    return 0;
}

## F 通配符匹配

In [ ]:
#include <stdio.h>
#include <string.h>
#include <stdbool.h>
#include <stdlib.h>

#define MAXLEN 100010
#define BASE 91138233  // 或者 131
typedef unsigned long long ull;

ull pow_arr[MAXLEN];

void init_pow(int max_len) {
    pow_arr[0] = 1;
    for (int i = 1; i <= max_len; i++) {
        pow_arr[i] = pow_arr[i-1] * BASE;
    }
}

typedef struct {
    char *str;          // 段字符串（可选，用于调试）
    int len;            // 段长度
    int q_count;        // ? 的数量
    int *q_pos;         // ? 在段中的位置（偏移）
    ull *q_weight;      // 每个?对应的权重 BASE^(len-1-pos)
    ull hash;           // 段哈希（普通字符贡献，?贡献0）
} Segment;

Segment segs[11];
int seg_cnt = 0;
bool start_star, end_star;


void build_segments(const char *pattern) {
    int pat_len = strlen(pattern);
    start_star = (pattern[0] == '*');
    end_star = (pattern[pat_len-1] == '*');
    
    int i = 0;
    while (i < pat_len) {
        if (pattern[i] == '*') {
            i++;
            continue;
        }
        int start = i;
        while (i < pat_len && pattern[i] != '*') i++;
        int seg_len = i - start;
        char *seg_str = (char*)malloc(seg_len + 1);
        strncpy(seg_str, pattern + start, seg_len);
        seg_str[seg_len] = '\0';
        
        Segment *seg = &segs[seg_cnt];
        seg->str = seg_str;
        seg->len = seg_len;
        seg->q_count = 0;

        for (int j = 0; j < seg_len; j++) {
            if (seg_str[j] == '?') seg->q_count++;
        }
        seg->q_pos = (int*)malloc(seg->q_count * sizeof(int));
        seg->q_weight = (ull*)malloc(seg->q_count * sizeof(ull));

        ull h = 0;
        int q_idx = 0;
        for (int j = 0; j < seg_len; j++) {
            char c = seg_str[j];
            ull weight = pow_arr[seg_len - 1 - j];
            if (c == '?') {
                seg->q_pos[q_idx] = j;
                seg->q_weight[q_idx] = weight;
                q_idx++;
            } else {
                ull val = c - 'a' + 1;
                h += val * weight;
            }
        }
        seg->hash = h;
        seg_cnt++;
    }
}

bool match_segment_fast(const char *filename, int len, int j, const Segment *seg, ull *h) {
    if (j + seg->len > len) return false;
    ull sub_hash = h[j + seg->len] - h[j] * pow_arr[seg->len];
    // 减去?位置上的字符贡献
    for (int k = 0; k < seg->q_count; k++) {
        int offset = seg->q_pos[k];
        char c = filename[j + offset];
        ull val = c - 'a' + 1;
        sub_hash -= val * seg->q_weight[k];
    }
    return sub_hash == seg->hash;
}

// 判断文件名是否匹配
bool is_match(const char *filename, const char *pattern) {
    int len = strlen(filename);
    if (seg_cnt == 0) return true; 
    
    ull h[MAXLEN];
    h[0] = 0;
    for (int i = 0; i < len; i++) {
        h[i+1] = h[i] * BASE + (filename[i] - 'a' + 1);
    }
    
    int pos = 0;
    int seg_idx = 0;
    
    if (!start_star) {
        if (!match_segment_fast(filename, len, 0, &segs[0], h)) return false;
        pos = segs[0].len;
        seg_idx = 1;
    } else {
        int found = -1;
        for (int j = 0; j + segs[0].len <= len; j++) {
            if (match_segment_fast(filename, len, j, &segs[0], h)) {
                found = j;
                break;
            }
        }
        if (found == -1) return false;
        pos = found + segs[0].len;
        seg_idx = 1;
    }
    
    // 处理后续段
    for (; seg_idx < seg_cnt; seg_idx++) {
        const Segment *seg = &segs[seg_idx];
        int found = -1;
        for (int j = pos; j + seg->len <= len; j++) {
            if (match_segment_fast(filename, len, j, seg, h)) {
                found = j;
                break;
            }
        }
        if (found == -1) return false;
        pos = found + seg->len;
    }
    
    // 检查结尾
    if (!end_star && pos != len) return false;
    return true;
}

int main() {
    char pattern[100010];
    int n;
    char filename[100010];
    
    scanf("%s", pattern);
    scanf("%d", &n);
    
    init_pow(100005);
    
    build_segments(pattern);
    
    for (int i = 0; i < n; i++) {
        scanf("%s", filename);
        puts(is_match(filename, pattern) ? "YES" : "NO");
    }
    
    for (int i = 0; i < seg_cnt; i++) {
        free(segs[i].str);
        free(segs[i].q_pos);
        free(segs[i].q_weight);
    }
    
    return 0;
}

## G 汉诺塔

In [ ]:
#include <stdio.h>

typedef long long ll;

ll f[4][35];   // f[x][i]: 从柱子x移动上面i个盘子所需步数
int g[4][35];  // g[x][i]: 移动后这些盘子所在的柱子（1:A,2:B,3:C）

int main() {
    int n;
    scanf("%d", &n);

    int vis[4] = {0}; // 是否已确定起始柱的第一次移动
    for (int i = 1; i <= 6; i++) {
        char op[3];
        scanf("%s", op);
        int x = op[0] - 'A' + 1;
        int y = op[1] - 'A' + 1;
        if (!vis[x]) {
            f[x][1] = 1;
            g[x][1] = y;
            vis[x] = 1;
        }
    }

    for (int i = 2; i <= n; i++) {
        for (int x = 1; x <= 3; x++) {
            int y = g[x][i-1];
            int z = 6 - x - y;
            if (g[y][i-1] == z) {
                g[x][i] = z;
                f[x][i] = f[x][i-1] + 1 + f[y][i-1];
            } else if (g[y][i-1] == x) {
                g[x][i] = y;
                f[x][i] = f[x][i-1] + 1 + f[y][i-1] + 1 + f[x][i-1];
            }
        }
    }

    printf("%lld\n", f[1][n]);
    return 0;
}

## H 马步距离

In [ ]:
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define OFFSET 10          // 偏移量，使下标非负
#define MAX 21             // 坐标范围 -10 ~ 10，加上偏移后 0~20

int dx[8] = {1, 1, -1, -1, 2, 2, -2, -2};
int dy[8] = {2, -2, 2, -2, 1, -1, 1, -1};
int dist[MAX][MAX];        // 步数表
int qx[MAX*MAX], qy[MAX*MAX]; // BFS队列

void bfs() {
    int head = 0, tail = 0;
    // 初始化距离为一个大数
    memset(dist, 0x3f, sizeof(dist));
    dist[0+OFFSET][0+OFFSET] = 0;
    qx[tail] = 0;
    qy[tail] = 0;
    tail++;
    
    while (head < tail) {
        int x = qx[head];
        int y = qy[head];
        head++;
        int cur = dist[x+OFFSET][y+OFFSET];
        for (int i = 0; i < 8; i++) {
            int nx = x + dx[i];
            int ny = y + dy[i];
            // 只处理范围内坐标 (-10..10)
            if (nx >= -10 && nx <= 10 && ny >= -10 && ny <= 10) {
                if (dist[nx+OFFSET][ny+OFFSET] > cur + 1) {
                    dist[nx+OFFSET][ny+OFFSET] = cur + 1;
                    qx[tail] = nx;
                    qy[tail] = ny;
                    tail++;
                }
            }
        }
    }
}

int main() {
    int xp, yp, xs, ys;
    scanf("%d %d %d %d", &xp, &yp, &xs, &ys);
    int dx = abs(xp - xs);
    int dy = abs(yp - ys);
    // 确保 dx >= dy
    if (dx < dy) {
        int t = dx; dx = dy; dy = t;
    }
    
    bfs();  // 预处理小范围步数表
    
    int ans = 0;
    // 贪心缩减到小范围
    while (dx > 10 || dy > 10) {
        if (dx > dy) {
            dx -= 2;
            dy -= 1;
        } else {
            dx -= 1;
            dy -= 2;
        }
        // 取绝对值（防止出现负数）
        dx = abs(dx);
        dy = abs(dy);
        ans++;
    }
    // 加上小范围内的最少步数
    ans += dist[dx+OFFSET][dy+OFFSET];
    printf("%d\n", ans);
    return 0;
}

## I 直方图最大矩形

In [ ]:
#include <stdlib.h>

int largestRectangleArea(int* heights, int heightsSize) {
    int* stack = (int*)malloc((heightsSize + 1) * sizeof(int));
    int top = -1;
    long long maxArea = 0;
    
    for (int i = 0; i <= heightsSize; ++i) {
        int curHeight = (i == heightsSize) ? 0 : heights[i];
        while (top != -1 && curHeight < heights[stack[top]]) {
            int h = heights[stack[top--]];
            int width = (top == -1) ? i : i - stack[top] - 1;
            long long area = (long long)h * width;
            if (area > maxArea) maxArea = area;
        }
        stack[++top] = i;
    }
    
    free(stack);
    return (int)maxArea;
}

## J 消防局的设立

In [ ]:
#include <stdio.h>
#include <string.h>

#define MAXN 1005

int fa[MAXN];          // 父节点
int dis[MAXN];         // 深度
int cover[MAXN];       // 是否已被覆盖
int son[MAXN][MAXN];   // 子节点列表
int son_cnt[MAXN];     // 子节点个数
int n;

int main() {
    scanf("%d", &n);
    // 初始化根节点
    fa[1] = 1;
    dis[1] = 0;
    memset(son_cnt, 0, sizeof(son_cnt));
    // 读入并建树
    for (int i = 2; i <= n; i++) {
        int x;
        scanf("%d", &x);
        fa[i] = x;
        dis[i] = dis[x] + 1;
        son[x][son_cnt[x]++] = i;
    }

    int ans = 0;
    memset(cover, 0, sizeof(cover));

    while (1) {
        // 寻找未被覆盖且深度最大的节点
        int k = 0, maxd = -1;
        for (int i = 1; i <= n; i++) {
            if (!cover[i] && dis[i] > maxd) {
                maxd = dis[i];
                k = i;
            }
        }
        if (k == 0) break;   // 所有节点都已覆盖

        // 在 k 的祖父处设立消防局
        int m = fa[fa[k]];   // 祖父节点
        ans++;

        // 覆盖以 m 为中心、距离不超过 2 的所有节点
        cover[m] = 1;
        cover[fa[m]] = 1;
        cover[fa[fa[m]]] = 1;

        // 覆盖 m 的所有子节点（距离 1）
        for (int j = 0; j < son_cnt[m]; j++) {
            int u = son[m][j];
            cover[u] = 1;
            // 覆盖 m 的孙子节点（距离 2）
            for (int l = 0; l < son_cnt[u]; l++) {
                cover[son[u][l]] = 1;
            }
        }

        // 覆盖 m 的父亲的所有子节点（距离 2）
        int mm = fa[m];
        for (int j = 0; j < son_cnt[mm]; j++) {
            cover[son[mm][j]] = 1;
        }
    }

    printf("%d\n", ans);
    return 0;
}